# LSTM CARAVAN — Google Colab (full Caravan replication, batched)

Colab-adapted version of `LSTM_CARAVAN.ipynb` that reproduces the **full** Caravan pretraining run from **[examples/configs/caravan.yml](../examples/configs/caravan.yml)** — all 17,847 basins in `data/basin_id/basins_caravan_pretrain.txt` (same list the cluster run uses), minus a small Colab-only gap explained below — without needing ~137GB of raw CSVs present on disk at once.

**Why batching was necessary (short version of a longer debugging story):** Colab's local disk is commonly only ~100–110GB, smaller than the ~137GB assembled raw dataset. Two earlier approaches both failed: staging the full raw dataset locally ran out of disk mid-extraction, and extracting it directly onto Google Drive's FUSE mount hit unexplained I/O errors (Drive is a known weak point for writing thousands of small files). Neither problem is really about *how* the data gets somewhere — it's that `hy2dl`'s dataset-processing step previously needed every basin's raw CSV simultaneously readable, no matter where it lived.

**What changed:** `hy2dl`'s `_create_zarr_dataset` (in `src/hy2dl/datasetzoo/basedataset.py`) now accepts an optional `gauge_id_batch` argument, letting it be called repeatedly against successive subsets of basins instead of all of them at once. This notebook downloads Caravan's 9 source archives once (~41GB, kept on local disk for the whole run), then processes basins in batches — one batch per source, except `hysets` (68GB alone, the single largest sub-dataset) which is split into 5 sub-batches of ~1,600 basins. For each batch: extract just those basins' raw CSVs from the already-downloaded archives, write them into the (persistent, incrementally-growing) zarr cache, delete the extracted CSVs, move to the next batch. Peak local disk stays bounded by whichever single batch is largest (~14GB) plus the ~41GB of cached archives — nowhere near the full 137GB.

The resulting zarr caches (training/validation/testing, a few GB each once compressed by processing rather than tens of GB of raw text) get archived to Google Drive as single tar files, the same safe single-large-file I/O pattern used elsewhere in this notebook — so a later session restores them directly instead of re-downloading and re-processing.

**Colab-only basin-count gap (343 `camelsaus` basins):** `data/basin_id/basins_caravan_pretrain.txt` itself lists 561 `camelsaus` basins and is *not* modified by this notebook - the cluster run depends on it staying at the full 17,847, and the cluster's pre-assembled `Caravan_all` dataset already has all 561 (someone sourced/converted the extra 339 separately, outside any public Caravan archive). But the public Caravan `core` archive this notebook downloads only ever converted CAMELS-AUS **v1** (222 catchments) into Caravan's CSV format - CAMELS-AUS **v2** (561 catchments, released later, see [Zenodo record 13350616](https://zenodo.org/records/13350616)) was checked in both the currently-used archive version (Zenodo 10968468, v1.4) and the newer v1.5 (Zenodo 14673536) and neither includes it. Since Colab can't reach the cluster's private storage, these 339 basins aren't obtainable here. `data/basin_id/camelsaus_unavailable_colab.txt` lists exactly which 339 IDs those are (confirmed by diffing the basin list against the real archive contents via `tar -tJf`, not guessed), and the bootstrap cell below excludes them *only* from this notebook's own run (17,847 → 17,504 basins for this Colab run specifically) by passing an explicit `gauge_id=` list to each `Dataset(...)` call rather than letting it read the unmodified file from disk. Every other `core`-archive source (`camels`, `camelsbr`, `camelscl`, `camelsgb`, `hysets`, `lamah`) was checked the same way and has full ID overlap - no other gaps.

**What's verified vs. what isn't:** the core batching mechanism (`_create_zarr_dataset(gauge_id_batch=...)`) was tested against real Caravan data on the cluster and produces byte-identical output to the original single-shot method. The download/extraction orchestration below reuses URLs and extraction mechanics (wildcard patterns for zip wrapper folders, `tar`/`unzip` include-filtering, and the `core` archive's build-path prefix) individually verified against the real archives, and the full pipeline has now been run live through the `core` archive's extraction path successfully. Remaining zip-only sources (`camelsde`, `camelsch`, `camelscz`, `camelsdk`, `camelses`, `il`, `lamahice`) still haven't been observed end-to-end in this notebook.

**Expect:** ~41GB download, well under half your disk in local use at any point, on the first run only. Estimate the total batched-processing time generously — 20 batches (15 single-source + 5 `hysets` sub-batches), each with its own download-once/extract/write/cleanup cycle. `core`-archive sources (`camels`, `camelsaus`, `camelsbr`, `camelscl`, `camelsgb`, `hysets`, `lamah`) extract much slower than the zip-based sources since `tar` has to sequentially decompress the xz stream up to wherever each source's entries sit, rather than zip's random-access central directory - likely a few hours total for the first run; every session after that just restores three zarr tars from Drive. Recommended: Colab Pro/Pro+ with an A100/L4 GPU.

## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
import sys
from pathlib import Path

%cd /content
if Path("/content/Hy2DL_new").exists():
    %cd /content/Hy2DL_new
    !git pull
else:
    !git clone https://github.com/sanikabaste/Hy2DL_new.git
    %cd /content/Hy2DL_new

!pip install -q -e .

# pip's editable install only takes effect for a *new* Python process (it's picked up via a .pth file that
# `site` processes at interpreter startup) - add src/ to sys.path directly so `import hy2dl` below works in
# this same running kernel too, without needing a runtime restart.
src_path = str(Path("/content/Hy2DL_new/src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Configure the experiment

Same `caravan.yml` as the cluster run (17,847 basins, minus 339 Colab-only-unavailable `camelsaus` basins - see the intro cell). `path_save_folder` and the three `path_dataset_*` zarr caches are redirected to Google Drive so both training checkpoints and the processed dataset survive a Colab disconnect.

In [ ]:
import datetime
import random
import shutil
import subprocess
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
import yaml

from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation import calculate_metrics, get_tester
from hy2dl.modelzoo import get_model
from hy2dl.training.basetrainer import BaseTrainer
from hy2dl.utils.config import Config

# caravan.yml's paths (e.g. "../data/basin_id/...") are written relative to examples/, matching how the
# cluster script (examples/caravan_training.py) runs - cd into examples/ before python. base_dir has to
# match that convention or Config resolves them one level too high (e.g. /content/data/... instead of
# /content/Hy2DL_new/data/...).
base_dir = (Path.cwd().resolve() / "examples")
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

with open("examples/configs/caravan.yml") as f:
    cfg_dict = yaml.safe_load(f)

cfg_dict["path_save_folder"] = "/content/drive/MyDrive/Hy2DL_Caravan_results"
cfg_dict["dataset_in_ram"] = False  # required for the batched/zarr processing path used below

config = Config(cfg_dict, base_dir=base_dir)
config.init_experiment()

config.path_dataset_training = config.path_save_folder / "dataset_training.zarr"
config.path_dataset_validation = config.path_save_folder / "dataset_validation.zarr"
config.path_dataset_testing = config.path_save_folder / "dataset_testing.zarr"

config.dump()

Dataset = get_dataset(config)
Tester = get_tester(config)

## 3. Download and batch-process the Caravan dataset

This is the long section. It downloads all 9 source archives once, processes basins in batches (writing each batch straight into the persistent zarr caches and discarding the raw CSVs afterward), and caches the finished zarr stores to Drive. On a later session, if all three zarr caches are already on Drive, this whole section reduces to three fast tar restores.

In [ ]:
CARAVAN_DIR = Path("data/Caravan_all").resolve()  # local scratch for raw CSVs — only ever holds the current batch
SCRATCH_ARCHIVES = Path("/content/_caravan_archives")  # compressed archives, kept for the whole run (~41GB)
EXTRACT_TMP = Path("/content/_caravan_extract_tmp")
DRIVE_ZARR_CACHE_DIR = Path("/content/drive/MyDrive/Hy2DL_Caravan_data")

for d in (CARAVAN_DIR / "attributes", CARAVAN_DIR / "timeseries" / "csv", SCRATCH_ARCHIVES):
    d.mkdir(parents=True, exist_ok=True)

# (archive key -> url, filename, exact expected size in bytes, archive kind, source names it contains)
# Every URL/size pair was verified against the dataset already staged on the cluster before being added here.
ARCHIVES = {
    "core": dict(
        url="https://zenodo.org/api/records/10968468/files/Caravan-csv.tar.xz/content",
        filename="Caravan-csv.tar.xz",
        size=23433742084,
        kind="tar",
        sources=["camels", "camelsaus", "camelsbr", "camelscl", "camelsgb", "hysets", "lamah"],
        # This archive's members carry the original packager's absolute build path (confirmed via
        # `tar -tJf`), not a bare "timeseries/csv/..." root like the other archives - e.g. a real member is
        # "usr/local/google/home/kratzert/Data/Caravan-csv/Caravan/timeseries/csv/camels/camels_...csv".
        # Every extraction pattern for this archive needs this prefix or tar reports "Not found in archive".
        path_prefix="usr/local/google/home/kratzert/Data/Caravan-csv/Caravan/",
    ),
    "grdc": dict(
        url="https://zenodo.org/api/records/15349031/files/GRDC_Caravan_extension_csv.zip/content",
        filename="GRDC_Caravan_extension_csv.zip",
        size=8842746230,
        kind="zip",
        sources=["grdc"],
    ),
    "camelsde": dict(
        url="https://zenodo.org/api/records/14755229/files/caravan_de.zip/content",
        filename="caravan_de.zip",
        size=6085830462,
        kind="zip",
        sources=["camelsde"],
    ),
    "camelsch": dict(
        url="https://zenodo.org/api/records/15025258/files/Caravan_extension_CH.zip/content",
        filename="Caravan_extension_CH.zip",
        size=545597282,
        kind="zip",
        sources=["camelsch"],
    ),
    "camelscz": dict(
        url="https://zenodo.org/api/records/17769325/files/Caravan-Extension-CZ.zip/content",
        filename="Caravan-Extension-CZ.zip",
        size=913172886,
        kind="zip",
        sources=["camelscz"],
    ),
    "camelsdk": dict(
        url="https://zenodo.org/api/records/15200118/files/Caravan_extension_DK.zip/content",
        filename="Caravan_extension_DK.zip",
        size=521600377,
        kind="zip",
        sources=["camelsdk"],
    ),
    "camelses": dict(
        url="https://zenodo.org/api/records/15040948/files/CAMELS-ES_v110.zip/content",
        filename="CAMELS-ES_v110.zip",
        size=402788602,
        kind="zip",
        sources=["camelses"],
    ),
    "il": dict(
        url="https://zenodo.org/api/records/15181680/files/Caravan_extension_Israel_Ver4.zip/content",
        filename="Caravan_extension_Israel_Ver4.zip",
        size=294290689,
        kind="zip",
        sources=["il"],
    ),
    "ausvic": dict(
        url="https://zenodo.org/api/records/20627056/files/caravan_ausvic.zip/content",
        filename="caravan_ausvic.zip",
        size=33652672,
        kind="zip",
        sources=["ausvic"],
    ),
    "lamahice": dict(
        # Hosted on HydroShare, not Zenodo. Note: the version there today (283,091,076 bytes) does not
        # byte-match the copy used to build the cluster's dataset (383,647,089 bytes) - HydroShare doesn't
        # version files the way Zenodo does, so this is very likely a content update on their end rather
        # than a broken link. Affects only 74 of 17,847 basins (0.4%).
        url="https://www.hydroshare.org/resource/86117a5f36cc4b7c90a5d54e18161c91/data/contents/Caravan_extension_lamahice.zip/",
        filename="Caravan_extension_lamahice.zip",
        size=283091076,
        kind="zip",
        sources=["lamahice"],
    ),
}
SOURCE_TO_ARCHIVE = {s: key for key, info in ARCHIVES.items() for s in info["sources"]}
HYSETS_BATCH_SIZE = 1600  # splits hysets' ~7,679 basins into ~5 sub-batches of roughly this size


def download_archive(key: str) -> Path:
    info = ARCHIVES[key]
    dest = SCRATCH_ARCHIVES / info["filename"]
    if dest.exists() and dest.stat().st_size == info["size"]:
        return dest
    print(f"  Downloading {info['filename']} ...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(dest), info["url"]], check=True)
    actual = dest.stat().st_size
    if actual != info["size"]:
        raise RuntimeError(
            f"{info['filename']}: downloaded {actual} bytes, expected {info['size']}. "
            "The upstream file may have changed — stopping rather than risk training on the wrong data."
        )
    return dest


def extract_files(archive_key: str, file_patterns: list[str], dir_patterns: list[str], dest_dir: Path) -> None:
    """Extract specific member paths from an archive into dest_dir. `file_patterns` are exact file paths (e.g.
    a single basin's CSV); `dir_patterns` are directory paths whose entire contents should be extracted (e.g.
    a source's attributes folder).
    """
    info = ARCHIVES[archive_key]
    archive_path = SCRATCH_ARCHIVES / info["filename"]
    dest_dir.mkdir(parents=True, exist_ok=True)
    if info["kind"] == "tar":
        # tar matches directory patterns as prefixes, so bare paths work for both files and directories -
        # except this archive's members are all rooted under path_prefix (see ARCHIVES["core"]), which has
        # to be prepended to every pattern or tar reports "Not found in archive" for all of them.
        prefix = info.get("path_prefix", "")
        patterns = [prefix + p for p in file_patterns + dir_patterns]
        result = subprocess.run(
            ["tar", "-xJf", str(archive_path), "-C", str(dest_dir), *patterns],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            free_gb = shutil.disk_usage("/content").free / 1e9
            raise RuntimeError(
                f"tar extraction from {archive_path.name} failed (exit {result.returncode}), "
                f"{free_gb:.1f}GB free on /content.\nstderr (last 4000 chars):\n{result.stderr[-4000:]}"
            )
    else:
        # zip archives wrap their content in a varying number of nested folders (0-2 levels observed across
        # sources), and unzip needs an exact path match rather than a prefix match, so: pass both the bare
        # pattern and a "*/"-prefixed one to match regardless of wrapper depth, and append "/*" to directory
        # patterns so their contents (not just the directory entry itself) get selected. unzip exits non-zero
        # if *any* given pattern fails to match, even when others succeed - since exactly one of each bare/
        # wrapped pair is expected to miss depending on the archive's actual wrapper depth, that's not treated
        # as fatal here; the caller verifies the files it actually needed landed instead.
        all_patterns = (
            file_patterns
            + [f"*/{p}" for p in file_patterns]
            + [f"{p}/*" for p in dir_patterns]
            + [f"*/{p}/*" for p in dir_patterns]
        )
        subprocess.run(["unzip", "-q", "-o", str(archive_path), *all_patterns, "-d", str(dest_dir)])


def merge_extracted(extract_root: Path, target_root: Path) -> list[str]:
    """Move every file under any 'attributes/<source>/' or 'timeseries/csv/<source>/' subtree found anywhere
    inside extract_root into the matching location under target_root, merging into (rather than overwriting)
    whatever's already there - safe to call repeatedly for the same source across sub-batches. Searches
    recursively so it finds these subtrees regardless of how deeply nested the archive's own wrapper path is
    (e.g. the core archive's path_prefix).
    """
    moved = []
    for attributes_dir in extract_root.rglob("attributes"):
        if attributes_dir.is_dir():
            for source_dir in attributes_dir.iterdir():
                if source_dir.is_dir():
                    dest = target_root / "attributes" / source_dir.name
                    dest.mkdir(parents=True, exist_ok=True)
                    for f in source_dir.iterdir():
                        shutil.move(str(f), str(dest / f.name))
                    moved.append(f"attributes/{source_dir.name}")
    for csv_dir in extract_root.rglob("csv"):
        if csv_dir.is_dir() and csv_dir.parent.name == "timeseries":
            for source_dir in csv_dir.iterdir():
                if source_dir.is_dir():
                    dest = target_root / "timeseries" / "csv" / source_dir.name
                    dest.mkdir(parents=True, exist_ok=True)
                    for f in source_dir.iterdir():
                        shutil.move(str(f), str(dest / f.name))
                    moved.append(f"timeseries/csv/{source_dir.name}")
    return moved


def fetch_batch(source: str, gauge_ids: list[str], need_attributes: bool) -> None:
    """Download (if needed) the archive for `source`, extract just `gauge_ids`' raw CSVs (plus that source's
    small attribute files, once) into CARAVAN_DIR, and verify they actually landed.
    """
    archive_key = SOURCE_TO_ARCHIVE[source]
    download_archive(archive_key)
    file_patterns = [f"timeseries/csv/{source}/{gid}.csv" for gid in gauge_ids]
    dir_patterns = [f"attributes/{source}"] if need_attributes else []
    if EXTRACT_TMP.exists():
        shutil.rmtree(EXTRACT_TMP)
    extract_files(archive_key, file_patterns, dir_patterns, EXTRACT_TMP)
    merge_extracted(EXTRACT_TMP, CARAVAN_DIR)
    shutil.rmtree(EXTRACT_TMP)

    missing = [gid for gid in gauge_ids if not (CARAVAN_DIR / "timeseries" / "csv" / source / f"{gid}.csv").exists()]
    if missing:
        raise RuntimeError(f"{source}: failed to extract {len(missing)} basin file(s), e.g. {missing[:5]}")
    if need_attributes and not any((CARAVAN_DIR / "attributes" / source).iterdir()):
        raise RuntimeError(f"{source}: failed to extract attributes")


def drop_batch(source: str, gauge_ids: list[str]) -> None:
    """Delete just-processed raw CSVs for this batch to free local disk before the next one."""
    for gid in gauge_ids:
        f = CARAVAN_DIR / "timeseries" / "csv" / source / f"{gid}.csv"
        f.unlink(missing_ok=True)


print(f"{len(ARCHIVES)} source archives defined, covering {sum(len(i['sources']) for i in ARCHIVES.values())} sources.")

In [ ]:
# Bootstrap: constructing a Dataset object reads gauge_id[0]'s raw data to infer schema (date frequency etc.),
# so make sure at least that one basin's file is available before constructing anything - regardless of
# whether we end up doing a fresh full download or restoring cached zarr stores from Drive below.
all_basins = Path(config.path_entities_training).read_text().split()

# Colab-only exclusion (see intro cell): 339 camelsaus basins aren't obtainable from any public Caravan
# archive. This does NOT modify basins_caravan_pretrain.txt on disk - the cluster run depends on that file
# staying at the full 17,847, and its pre-assembled Caravan_all already has all of them. gauge_id is passed
# explicitly to each Dataset(...) below (rather than left to read config.path_entities_* from disk itself)
# so this exclusion actually takes effect - hy2dl hard-fails if any entity list references a gauge_id absent
# from the zarr store (see basedataset.py's "not present in the dataset" check).
colab_unavailable = set(Path("data/basin_id/camelsaus_unavailable_colab.txt").read_text().split())
if colab_unavailable:
    before = len(all_basins)
    all_basins = [b for b in all_basins if b not in colab_unavailable]
    print(
        f"Colab-only limitation: excluding {before - len(all_basins)} camelsaus basins not available in any "
        f"public Caravan archive ({len(all_basins)} of {before} basins remain for this run)."
    )

bootstrap_basin = all_basins[0]
bootstrap_source = bootstrap_basin.split("_")[0].lower()
print(f"Bootstrapping with {bootstrap_basin} (source={bootstrap_source})...")
fetch_batch(bootstrap_source, [bootstrap_basin], need_attributes=True)

training_dataset = Dataset(cfg=config, time_period="training", gauge_id=all_basins)
validation_dataset = Dataset(cfg=config, time_period="validation", gauge_id=all_basins)
testing_dataset = Dataset(cfg=config, time_period="testing", gauge_id=all_basins)
for ds, period in ((training_dataset, "training"), (validation_dataset, "validation"), (testing_dataset, "testing")):
    ds.path_dataset = getattr(config, f"path_dataset_{period}")
print("Dataset objects constructed.")

In [ ]:
zarr_paths = [training_dataset.path_dataset, validation_dataset.path_dataset, testing_dataset.path_dataset]
zarr_cache_tars = [DRIVE_ZARR_CACHE_DIR / f"{p.name}.tar" for p in zarr_paths]

if all(t.exists() for t in zarr_cache_tars):
    print("Found cached zarr datasets on Drive. Restoring locally instead of re-processing...")
    for path, tar_path in zip(zarr_paths, zarr_cache_tars):
        if path.exists():
            shutil.rmtree(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-xf", str(tar_path), "-C", str(path.parent)], check=True)
    NEED_PROCESSING = False
    print("Restored. The batched-processing cell below will skip itself.")
else:
    NEED_PROCESSING = True
    print("No complete cache found on Drive — will download and batch-process from scratch.")

In [ ]:
if NEED_PROCESSING:
    # Group all basins by source, splitting hysets into sub-batches (it alone is 68GB extracted - too large
    # to extract in one piece alongside the cached archives without risking local disk exhaustion).
    basins_by_source = defaultdict(list)
    for b in all_basins:
        basins_by_source[b.split("_")[0].lower()].append(b)

    batches = []  # (label, source, gauge_ids)
    for source, ids in basins_by_source.items():
        if source == "hysets":
            for i in range(0, len(ids), HYSETS_BATCH_SIZE):
                chunk = ids[i : i + HYSETS_BATCH_SIZE]
                batches.append((f"hysets[{i}:{i + len(chunk)}]", source, chunk))
        else:
            batches.append((source, source, ids))

    print(f"{len(batches)} batches covering {sum(len(b[2]) for b in batches)} basins.\n")

    seen_sources = set()
    for i, (label, source, gauge_ids) in enumerate(batches):
        is_last = i == len(batches) - 1
        print(f"=== Batch {i + 1}/{len(batches)}: {label} ({len(gauge_ids)} basins) ===")

        need_attributes = source not in seen_sources
        seen_sources.add(source)
        fetch_batch(source, gauge_ids, need_attributes=need_attributes)

        for ds in (training_dataset, validation_dataset, testing_dataset):
            ds._create_zarr_dataset(gauge_id_batch=gauge_ids, consolidate=is_last)

        drop_batch(source, gauge_ids)
        print()

    print("All batches processed. Cleaning up cached archives...")
    shutil.rmtree(SCRATCH_ARCHIVES, ignore_errors=True)
    shutil.rmtree(CARAVAN_DIR, ignore_errors=True)

    print("Caching zarr datasets to Drive (one-time cost, single large files)...")
    DRIVE_ZARR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for path, tar_path in zip(zarr_paths, zarr_cache_tars):
        subprocess.run(["tar", "-cf", str(tar_path), "-C", str(path.parent), path.name], check=True)
    print("Done. Future sessions will restore from Drive instead of repeating this section.")
else:
    print("Skipping — using zarr datasets restored from Drive cache.")

## 4. Finish dataset setup and train (resumable)

`setup_dataset()` finds the zarr caches built (or restored) above and loads them directly, then proceeds through the normal attribute-processing, sample-validation, and scaler steps — none of which need the raw CSVs anymore.

**If your Colab session disconnects:** reconnect, re-run every cell above, then re-run the training cell — it detects the last checkpoint saved on Drive and continues from `last_epoch + 1` instead of restarting.

In [ ]:
training_dataset.setup_dataset()
trainer = BaseTrainer(cfg=config, training_dataset=training_dataset)

In [ ]:
validation_dataset.setup_dataset(check_nan=False, path_scaler=config.path_save_folder / "scaler.yml")
tester_validation = Tester(cfg=config, evaluation_dataset=validation_dataset)

In [ ]:
model_dir = config.path_save_folder / "model"
completed_epochs = sorted(int(p.name.rsplit("_", 1)[1]) for p in model_dir.glob("model_epoch_*"))
start_epoch = 1
if completed_epochs:
    last_epoch = completed_epochs[-1]
    trainer.model.load_state_dict(
        torch.load(model_dir / f"model_epoch_{last_epoch}", map_location=config.device)
    )
    trainer.optimizer.update_optimizer_lr(epoch=last_epoch + 1)
    start_epoch = last_epoch + 1
    print(f"Resuming training from epoch {start_epoch} (found checkpoint for epoch {last_epoch}).")

if start_epoch > config.epochs:
    print(f"All {config.epochs} epochs already completed.")
else:
    validation_headers = "".join([f"{m:^10}|" for m in config.validation_metric])
    config.logger.info("Training model".center(60, "-"))
    config.logger.info(f"{'':^16}|{'Training':^21}|{'Validation':^{(11 * len(config.validation_metric)) + 10}}|")
    config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{validation_headers}{'Time':^10}|")

    total_time = time.time()
    for epoch in range(start_epoch, config.epochs + 1):
        trainer.train_model(epoch=epoch)  # Training
        tester_validation.validate_model(model=trainer.model, epoch=epoch)  # Validation
        config.logger.info(trainer.report + tester_validation.validation_report)  # report

    config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")
    shutil.rmtree(tester_validation.path_zarr, ignore_errors=True)  # delete validation results

## 5. Test and evaluate

In [ ]:
# Reconstruct the model from the last saved epoch and evaluate it on the testing period
model = get_model(config).to(config.device)
model.load_state_dict(
    torch.load(config.path_save_folder / "model" / f"model_epoch_{config.epochs}", map_location=config.device)
)

testing_dataset.setup_dataset(check_nan=False, path_scaler=config.path_save_folder / "scaler.yml")
tester_testing = Tester(cfg=config, evaluation_dataset=testing_dataset)

config.logger.info("Testing model...")
testing_time = time.time()
tester_testing.evaluate_model(model=model)
config.logger.info("Testing completed.")
config.logger.info(f"Total testing time: {datetime.timedelta(seconds=int(time.time() - testing_time))}\n")

test_results = xr.open_zarr(tester_testing.path_zarr)
testing_metrics = calculate_metrics(ds_results=test_results, metric_name=config.testing_metrics)
testing_metrics.to_zarr(config.path_save_folder / "testing_metrics.zarr", mode="w")

In [ ]:
# Loss testing
target_of_interest = random.sample(list(testing_metrics.feature.values), 1)[0]
test_metric = testing_metrics.sel(feature=target_of_interest, metric="nse").round(3).T.to_pandas().dropna()
# Plot the histogram
plt.figure(figsize=(10, 5))
plt.hist(test_metric, bins=np.linspace(0.0, 1.0, 11).tolist())
# Add NSE statistics to the plot
plt.text(
    0.01,
    0.8,
    (
        f"Mean: {'%.2f' % test_metric.mean():>7}\n"
        f"Median: {'%.2f' % test_metric.median():>0}\n"
        f"Max: {'%.2f' % test_metric.max():>9}\n"
        f"Min: {'%.2f' % test_metric.min():>10}"
    ),
    transform=plt.gca().transAxes,
    bbox=dict(facecolor="white", alpha=0.5),
)

# Format plot
plt.xlabel("NSE", fontsize=12, fontweight="bold")
plt.ylabel("Frequency", fontsize=12, fontweight="bold")
plt.title(f"NSE histogram for: {target_of_interest}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot simulated and observed discharges
basin_to_analyze = random.sample(list(test_results.gauge_id.values), 1)[0]
y_sim = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_sim"].compute().values
y_obs = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_obs"].compute().values

plt.figure(figsize=(15, 7.5))
plt.plot(y_obs, label="observed", color=color_palette["observed"])
plt.plot(y_sim, label="simulated", alpha=0.5, color=color_palette["simulated"])

# Format plot
plt.xlabel("Date", fontsize=12, fontweight="bold")
plt.ylabel(target_of_interest, fontsize=12, fontweight="bold")
plt.title(f"Results for gauge_id: {basin_to_analyze}", fontsize=16, fontweight="bold")
plt.tick_params(axis="both", which="major", labelsize=12)
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.show()